In [1]:
import cv2
import numpy as np
import pandas as pd
import joblib
import os
from datetime import datetime
from mtcnn import MTCNN
from keras_facenet import FaceNet

In [2]:
model = joblib.load("svm_model.pkl")
encoder = joblib.load("label_encoder.pkl")

In [3]:
embedder = FaceNet()
detector = MTCNN()

Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "C:\Users\HP\anaconda3\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "C:\Users\HP\anaconda3\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "C:\Users\HP\anaconda3\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.


In [4]:
today = datetime.now().strftime("%Y-%m-%d")

attendance_file = f"Attendance_{today}.csv"

if not os.path.exists(attendance_file):

    df = pd.DataFrame(columns=["Name","Date","Time"])

    df.to_csv(attendance_file,index=False)

print("Attendance File Ready")

Attendance File Ready


In [5]:
def mark_attendance(name):

    df = pd.read_csv(attendance_file)

    if name not in df["Name"].values:

        now = datetime.now()

        date = now.strftime("%Y-%m-%d")

        time = now.strftime("%H:%M:%S")

        new_data = pd.DataFrame({

            "Name":[name],
            "Date":[date],
            "Time":[time]

        })

        df = pd.concat([df,new_data],ignore_index=True)

        df.to_csv(attendance_file,index=False)

        print(name,"Attendance Marked")

In [ ]:
cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    faces = detector.detect_faces(rgb)

    for face in faces:

        x, y, w, h = face["box"]

        x = max(0, x)
        y = max(0, y)

        crop = rgb[y:y+h, x:x+w]

        if crop.size == 0:
            continue

        crop = cv2.resize(crop,(160,160))

        crop = np.expand_dims(crop,axis=0)

        embedding = embedder.embeddings(crop)[0]

        probabilities = model.predict_proba([embedding])[0]

        prediction = np.argmax(probabilities)

        confidence = probabilities[prediction]

        if confidence < 0.75:

            name = "Unknown Person"

            color = (0,0,255)

        else:

            name = encoder.inverse_transform([prediction])[0]

            color = (0,255,0)

            mark_attendance(name)

        cv2.rectangle(frame,(x,y),(x+w,y+h),color,2)

        cv2.putText(frame,

                    f"{name} {confidence:.2f}",

                    (x,y-10),

                    cv2.FONT_HERSHEY_SIMPLEX,

                    0.8,

                    color,

                    2)

    cv2.imshow("Attendance System",frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()

cv2.destroyAllWindows()